# factMemberRevenueGap Pipeline Orchestrator
Orchestrates the ingestion, gap engine, and conformed dimension loading sequence to write the final Platinum fact table.

In [ ]:
# Setup environment and spark session
try:
    dbutils
except NameError:
    import os, sys
    from pathlib import Path
    curr_path = Path(os.getcwd()).resolve()
    project_root = None
    for _ in range(5):
        if (curr_path / "local_setup.py").exists():
            project_root = curr_path
            break
        curr_path = curr_path.parent
    if project_root:
        sys.path.append(str(project_root))
        from local_setup import spark, dbutils, display
    else:
        print("Warning: Could not locate local_setup.py in parent directories!")


In [ ]:
import os
import sys
from pathlib import Path

ROOT_DIR = Path(os.getcwd()).parent
sys.path.append(str(ROOT_DIR))

try:
    dbutils.widgets.text("ClientContainer", "274", "Client Container / Catalog Name")
    client_container = dbutils.widgets.get("ClientContainer").strip()
except Exception:
    client_container = "274"


In [ ]:
def trigger_gap_engine(client_container_val: str):
    """Triggers the Risk Adjustment Gap Engine (Silver layer)"""
    notebook_path = f"./Silver/Notebooks/MemberRevenueGaps"
    print(f"\n=== Triggering Gap Engine: {notebook_path} ===")
    dbutils.notebook.run(notebook_path, 600, {"ClientContainer": client_container_val})

def trigger_platinum_fact_load(client_container_val: str):
    """Loads conformed dimensions and merges them into factMemberRevenueGap (Platinum)"""
    notebook_path = f"./Gold/Notebooks/GenericSubGroupProcessing"
    config_path = f"./Gold/Schema/factMemberRevenueGap.json"
    print(f"\n=== Triggering Platinum Fact Load: {notebook_path} ===")
    dbutils.notebook.run(notebook_path, 600, {
        "ClientContainer": client_container_val,
        "SubGroupConfigPath": config_path
    })


In [ ]:
if __name__ == "__main__":
    # Run the orchestration sequence
    trigger_gap_engine(client_container)
    trigger_platinum_fact_load(client_container)
    print("\nfactMemberRevenueGap pipeline execution completed successfully!")
